**Imports for Testing**

In [1]:
import unittest
import numpy as np
import itertools
from scipy.optimize import milp
from scipy.optimize import LinearConstraint
from scipy.optimize import Bounds

**Set Up a Brute Force Implementation for Additional Testing**

In [2]:
def brute_force_solution(names, project_names, rankings):
    # Get student names, project names, and student rankings for optimization.
    num_students = len(names)
    num_projects = len(project_names)

    # Make a matrix of values just like in the MILP approach.
    value = np.zeros((num_students, num_projects))
    for i in range(num_students):
        for rank_pos in range(rankings.shape[1]):  # 4 positions/picks
            project_index = int(rankings[i, rank_pos]) - 1
            value[i, project_index] = max(value[i, project_index], num_projects - rank_pos)
            
    # Maximum amounts of students in each capstone group 
    # (equal number of students in groups and remainder of students spread out to first few available projects).
    base = num_students // num_projects
    remainder = num_students % num_projects

    capacities = []
    for i in range(num_projects):
        if i < remainder: capacities.append(base + 1)
        else: capacities.append(base)

    capacities = np.array(capacities)

    # Most optimal solution/score.
    best_score = -np.inf
    best_assignment = None

    # Run through all assignments to find optimal options.
    for assignment in itertools.product(range(num_projects), repeat=num_students):
        # Set up values for a score based matrix.
        counts = np.zeros(num_projects, dtype=int)
        valid = True
        score = 0

        # Go through each assignment and calculate best possible scores. 
        for i, proj in enumerate(assignment):
            counts[proj] += 1
            if counts[proj] > capacities[proj]:
                valid = False
                break
            score += value[i, proj]

        # Assign best scores and related assignments.
        if valid and score > best_score:
            best_score = score
            best_assignment = assignment

    # Convert answer to a matrix form. 
    x = np.zeros((num_students, num_projects), dtype=int)
    for i, proj in enumerate(best_assignment):
        x[i, proj] = 1

    return x, best_score

**MILP Optimization Method Used in Project (Testing this Method)**

In [3]:
# Note: Not including data handling as it exists in the main code's method as it is not relevant to the optimization method itself, 
# only to handling input data.
def run_optimization(names, project_names, rankings):
    # Get student names, project names, and student rankings for optimization. 
    num_students = len(names)
    num_projects = len(project_names)

    # Higher preference rankings get higher values (1 being highest value).
    value = np.zeros((num_students, num_projects))

    for i in range(num_students):
        for rank_pos in range(rankings.shape[1]):  # 4 positions/picks
            project_index = int(rankings[i, rank_pos]) - 1
            if 0 <= project_index < num_projects:
                value[i, project_index] = max(value[i, project_index], num_projects - rank_pos)

    # Maximum amounts of students in each capstone group 
    # (equal number of students in groups and remainder of students spread out to first few available projects).
    base = num_students // num_projects
    remainder = num_students % num_projects

    capacities = []
    for i in range(num_projects):
        if i < remainder: capacities.append(base + 1)
        else: capacities.append(base)

    capacities = np.array(capacities)

    # Objective Function: Negate values for maximizing student preferences per each project.
    c = -value.flatten()

    # List to store constraints. 
    constraints = []

    # Equality Constraint Matrix
    A_eq_students = np.zeros((num_students, num_students * num_projects))

    # Each row is a student and each student is constrained to be a part of one project.
    for i in range(num_students):
        for j in range(num_projects):
            A_eq_students[i, i * num_projects + j] = 1

    # Each student assigned to one project.
    b_eq_students = np.ones(num_students)

    # Add equality constraints to constraint list.
    constraints.append(LinearConstraint(A_eq_students, b_eq_students, b_eq_students))

    # Inequality Constraint Matrix
    A_ub_projects = np.zeros((num_projects, num_students * num_projects))

    # Each column is a project and counts how many students are attached to it.
    for j in range(num_projects):
        for i in range(num_students):
            A_ub_projects[j, i * num_projects + j] = 1

    # Add inequality constraints to constraint list.
    constraints.append(LinearConstraint(A_ub_projects, -np.inf, capacities))

    # Each variable is binary: 0 not in group, 1 is + integrality is binary
    bounds = Bounds(0, 1)
    integrality = np.ones(num_students * num_projects, dtype=int)

    # Run the MILP optimization method. Reshape the solution into a matrix. Return error if optimization fails.
    result = milp(c=c, constraints=constraints, integrality=integrality, bounds=bounds)

    if result.x is None:
        raise ValueError(f"Optimization failed: {result.message}")
    
    assert result.success
    return result.x.reshape((num_students, num_projects)).astype(int), capacities

**Unit Tests on Constraints, Objective Function, a Stress Test, and a Comparison to Brute Force for the MILP Method**

In [4]:
class TestOptimization(unittest.TestCase):

    # Test that each student is only assigned to one project (equality constraint).
    def test_each_student_assigned_once(self):
        # Values used to test optimization.
        names = ["A", "B", "C"]
        projects = ["P1", "P2", "P3", "P4"]
        rankings = np.array([
            [1, 2, 1, 2],
            [1, 2, 1, 2],
            [2, 1, 2, 1]
        ])

        # Run optimization on the test values.
        x, capacities = run_optimization(names, projects, rankings)

        # Assert binary outputs (yes or no for each project).
        self.assertTrue(np.all((x == 0) | (x == 1)))

        # Assert that each student is assigned to exactly one project.
        for row in x:
            self.assertEqual(np.sum(row), 1)

    # Test that group capacities are being followed (inequality constraint).
    def test_capacity_constraints(self):
        # Values used to test optimization.
        names = ["A", "B", "C", "D"]
        projects = ["P1", "P2"]
        rankings = np.array([
            [1, 2],
            [1, 2],
            [1, 2],
            [2, 1]
        ])

        # Run optimization on the test values.
        x, capacities = run_optimization(names, projects, rankings)

        # Assert binary outputs (yes or no for each project).
        self.assertTrue(np.all((x == 0) | (x == 1)))

        # Count how many students are in a project.
        project_counts = np.sum(x, axis=0)

        # Make sure all groups have a number of students less than or equal to maximum capacity of students.
        for j in range(len(projects)):
            self.assertLessEqual(project_counts[j], capacities[j])

    # Test a situation where the optimal cases are known (objective function).
    def test_known_optimal_case(self):
        # Values used to test optimization.
        names = ["A", "B"]
        projects = ["P1", "P2"]
        rankings = np.array([
            [1, 2],  # A will prefer P1
            [2, 1]   # B will prefer P2
        ])

        # Run optimization on the test values.
        x, _ = run_optimization(names, projects, rankings)

        # Assert binary outputs (yes or no for each project).
        self.assertTrue(np.all((x == 0) | (x == 1)))

        # Assert that Student A gets P1 and Student B gets P2.
        self.assertEqual(x[0, 0], 1)
        self.assertEqual(x[1, 1], 1)

    # Test the optimization method using random inputs (stress test).
    def test_random_inputs(self):
        
        # Test several sets of random rankings values for a group of 6 students looking to get into 3 projects.
        for seed in range(5):
            # Use a random seed each time.
            np.random.seed(seed)

            num_students = 6
            num_projects = 3

            names = [f"S{i}" for i in range(num_students)]
            projects = [f"P{j}" for j in range(num_projects)]

            # Get random ranking values.
            rankings = np.array([
                np.random.permutation(np.arange(1, num_projects + 1))
                for _ in range(num_students)
            ])

            # Run optimization on the test values.
            x, capacities = run_optimization(names, projects, rankings)

            # Assert binary outputs (yes or no for each project).
            self.assertTrue(np.all((x == 0) | (x == 1)))

            # Check for feasibility in these random stress tests (check it still follows constraints).
            for row in x:
                self.assertEqual(np.sum(row), 1)

            # Make sure all groups have a number of students less than or equal to maximum capacity of students.
            project_counts = np.sum(x, axis=0)
            for j in range(num_projects):
                self.assertLessEqual(project_counts[j], capacities[j])

    # Test if brute force optimization gets the same results as the MILP optimization used for the project.
    def test_against_bruteforce(self):
        # Values used to test optimization.
        names = ["A","B","C","D"]
        projects = ["P1","P2","P3"]
        rankings = np.array([
            [1,2,3,1],
            [1,2,3,1],
            [2,1,3,2],
            [3,1,2,3]
        ])

        # Run MILP optimization and brute force optimization.
        x_milp, _ = run_optimization(names, projects, rankings)
        x_brute, brute_score = brute_force_solution(names, projects, rankings)

        # Compute the MILP method score similar to brute force method above.
        def compute_score(x):
            num_students, num_projects = x.shape
            score = 0
            for i in range(num_students):
                for j in range(num_projects):
                    if x[i, j] == 1:
                        for rank_pos in range(rankings.shape[1]):
                            if rankings[i, rank_pos] - 1 == j:
                                score += num_projects - rank_pos
            return score

        # Note: Score is essentially maximum value found for a project 
        # rather than what the best assignment is for each student.
        milp_score = compute_score(x_milp)

        # Make sure optimization method scores are equal to validate MILP approach.
        self.assertEqual(milp_score, brute_score)

# Load and print whether or not tests succeed.
load = unittest.TestLoader().loadTestsFromTestCase(TestOptimization)
result = unittest.TextTestRunner(verbosity=2).run(load)


test_against_bruteforce (__main__.TestOptimization.test_against_bruteforce) ... ok
test_capacity_constraints (__main__.TestOptimization.test_capacity_constraints) ... ok
test_each_student_assigned_once (__main__.TestOptimization.test_each_student_assigned_once) ... ok
test_known_optimal_case (__main__.TestOptimization.test_known_optimal_case) ... ok
test_random_inputs (__main__.TestOptimization.test_random_inputs) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.082s

OK
